# Imports

In [ ]:
import requests
from pathlib import Path
import os
from math import radians, cos, sin, asin, sqrt
from itertools import permutations
import time
import pandas as pd
from datetime import datetime, timedelta

import geopandas as gpd
import osmium
from shapely.geometry import LineString, Point
import re
import matplotlib.pyplot as plt
import networkx as nx

# Definition of constants, API keys, helper function

In [ ]:
BASE_DB_API = "https://apis.deutschebahn.com/db-api-marketplace/apis/"
STOP_PLACES_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stop-places"
RIS_STATION_URL = BASE_DB_API + "ris-station/v1/stations/"

V6_TRANSPORT_BASE_URL = "https://v6.db.transport.rest"

ROOT_DIR = Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = ROOT_DIR / "Raw_Data"
PROCESSED_DATA_DIR = ROOT_DIR / "Processed_Data"

scraping_raw_output_file = RAW_DATA_DIR / "train_routes_scraped.csv"
scraping_cleaned_output_file = PROCESSED_DATA_DIR / "train_routes_scraped.csv"

DB_CLIENT_ID = "a9f83c55d26c3ee7f48f4ce887ec2a57"
DB_API_KEY = "422cac21a0a83876c75efb8806589ea0"

header_ris = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/vnd.de.db.ris+json"
}

SCRAPING_RATE_LIMIT_SLEEP = 1.0  # Seconds to sleep between requests to respect 100 req/min

def haversine(lat1, lon1, lat2, lon2):
    """ Calculate distance in km between two points """
    R = 6371 # Earth radius
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    a = sin(dLat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dLon/2)**2
    return 2 * R * asin(sqrt(a))

#  Railway stations selection & filtering

First, match train station with selected cities from simplemaps ( Processed_Data/cities_500K). Later for matched cities retrieve required timetables.

Matching should be done geospatially

In [ ]:
cities_data = pd.read_csv(PROCESSED_DATA_DIR / "cities_500K.csv")
cities_data.rename(columns={"name_city":"name", "lat_city":"latitude", "lng_city":"longitude","country":"iso_code"}, inplace=True)
cities_data.head()

# Define functions to filter cities

In [ ]:
def get_stations_api_stop_places(cities, base_url = STOP_PLACES_URL, limit = 3):
    """
    Retrieves station information from the Deutsche Bahn API based on the station name.

    Keyword arguments:
    cities -- DataFrame containing city information
    base_url -- Base URL for the API
    limit -- Maximum number of stations to retrieve per city
    Return: DataFrame of stations for given cities
    """

    retrieved_stations = pd.DataFrame(columns=['city_name', 'eva_id', 'station_name', 'latitude', 'longitude'])

    for _, row in cities.iterrows():
        city_name = row['name']
        response = None
        params = {
            "sortBy": "RELEVANCE",
            "onlyActive": "true",
            "withSynonyms": "true",
            "latitude": row['latitude'],
            "longitude": row['longitude'],
            "limit": limit
        }
        success = False
        retries = 0

        while not success and retries < 3:
            response = requests.get(url=f"{base_url}/by-name/{city_name}", params=params, headers=header_ris)

            if response.status_code == 429:
                print(f"Rate limit reached. Sleeping for 1s...")
                time.sleep(1)
                retries += 1
                continue

            if response.status_code == 200:
                stations = response.json().get('stopPlaces', [])
                print(f"City: {city_name}, Stations Found: {len(stations)}")
                for station in stations:
                    latitude = float(station.get('position').get('latitude'))
                    longitude = float(station.get('position').get('longitude'))

                    if haversine(row['latitude'], row['longitude'], latitude, longitude) > 10:
                        print(f"Skipping station {station.get('names').get('DE').get('nameLong')} due to distance.")
                        continue

                    retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({
                        'city_name': [city_name],
                        'eva_id': [str(station.get('evaNumber'))],
                        'station_name': [str(station.get('names').get('DE').get('nameLong'))],
                        'latitude': [float(station.get('position').get('latitude'))],
                        'longitude': [float(station.get('position').get('longitude'))]
                    })])
                success = True
            else:
                print(f"Error retrieving stations for city {city_name}: {response.status_code}")
                break

            time.sleep(0.09)  # To respect API rate limits

    return retrieved_stations

# Get first station for every city

In [ ]:
cities_stations = get_stations_api_stop_places(cities_data, limit=1)

# Inspect for which cities stations have and have not been found

## No stations found

In [ ]:
cities_no_station = pd.merge(cities_data, cities_stations, how='outer',left_on=['name'], right_on=['city_name'], indicator=True).query('_merge == "left_only"')
cities_no_station.reset_index(inplace=True)
cities_no_station

## Stations found

In [ ]:
cities_with_stations = pd.merge(cities_data, cities_stations, how='inner',left_on=['name'], right_on=['city_name'])
cities_with_stations

# Definition of necessary functions for route scraping

In [ ]:
def get_next_representative_weekday():
    """
    Finds the next Tuesday or Wednesday.
    Mid-week days have the most consistent 'standard' schedules.
    """
    now = datetime.now()
    # 0=Mon, 1=Tue, 2=Wed...
    days_ahead = (1 - now.weekday() + 7) % 7 # Target Tuesday
    if days_ahead == 0: days_ahead = 7 # If today is Tuesday, get next week

    target_date = now + timedelta(days=days_ahead)
    # set to 08:00 AM - Peak morning traffic usually has the best connections
    return target_date.replace(hour=8, minute=0, second=0, microsecond=0)

def fetch_fastest_connection(origin_id, dest_id):
    """
    Makes a SINGLE smart request to find the fastest connection.
    Retrieves 5 options and picks the minimum duration.
    """
    target_date = get_next_representative_weekday()

    params = {
        "from": origin_id,
        "to": dest_id,
        "departure": target_date.isoformat(),
        "results": 5,           # Get 5 options in ONE request
        "national": "true",     # Prefer High Speed
        "nationalExpress": "true",
        "transfers": 4          # Allow complex routes if they are faster
    }

    # Retry logic for stability
    for attempt in range(3):
        try:
            response = requests.get(f"{V6_TRANSPORT_BASE_URL}/journeys", params=params, timeout=10)

            if response.status_code == 200:
                data = response.json()
                journeys = data.get('journeys', [])

                if not journeys:
                    return None

                # Optimization: Process all 5 results locally to find the minimum
                min_duration = float('inf')
                best_journey = None

                for journey in journeys:
                    if not journey.get('legs'): continue

                    dep = datetime.fromisoformat(journey['legs'][0]['departure'])
                    arr = datetime.fromisoformat(journey['legs'][-1]['arrival'])
                    duration = (arr - dep).total_seconds() / 60

                    if duration < min_duration:
                        min_duration = duration
                        best_journey = journey
                        best_journey['calculated_duration'] = int(duration)

                # Extract details from the winner
                legs = best_journey['legs']
                train_names = [l.get('line', {}).get('name', '') for l in legs if l.get('mode') == 'train']

                time.sleep(SCRAPING_RATE_LIMIT_SLEEP) # Respect limits
                return {
                    "origin_id": origin_id,
                    "destination_id": dest_id,
                    "origin_name": legs[0]['origin']['name'],
                    "destination_name": legs[-1]['destination']['name'],
                    "min_duration_minutes": best_journey['calculated_duration'],
                    "transfers": len(legs) - 1,
                    "trains": ", ".join(filter(None, train_names)),
                    "check_date": target_date.strftime("%Y-%m-%d")
                }

            elif response.status_code == 429:
                time.sleep(3 * (attempt + 1)) # Backoff
            elif response.status_code >= 500:
                time.sleep(2)
            else:
                return None

        except Exception as e:
            time.sleep(1)

    return None

def get_efficient_network_data(eva_ids):
    # use permutations because A->B might be slightly different than B->A
    # (e.g. connections matching up)
    pairs = list(permutations(eva_ids, 2))

    results = []

    for i, (origin, dest) in enumerate(pairs):
        print(f"[{i+1}/{len(pairs)}] {origin} -> {dest}...", end=" ", flush=True)

        data = fetch_fastest_connection(origin, dest)

        if data:
            print(f"found: {data['min_duration_minutes']} min")
            results.append(data)
        else:
            print(f"no route found")

    return pd.DataFrame(results)

# Define function to Scrape API with intermittent saves to prevent data loss

In [ ]:
def scrape_train_routes_with_checkpoints(eva_ids, batch_size = 10):
    """
    scrapes train routes and persists to CSV every batch_size API calls.

    Args:
        eva_ids: List of station EVA IDs to query
        batch_size: Save to CSV after this many successful API calls (default: 10)
    """

    processed_pairs = set()
    if scraping_raw_output_file.exists():
        existing_df = pd.read_csv(scraping_raw_output_file)
        processed_pairs = set(zip(existing_df['origin_id'], existing_df['destination_id']))
        print(f"found {len(processed_pairs)} existing routes, resuming scraping..")
    else:
        print(f"no existing routes found, created new output file {scraping_raw_output_file}")

    pairs = list(permutations(eva_ids, 2))
    pending_pairs = [p for p in pairs if p not in processed_pairs]

    batch_results = []
    api_calls_count = 0
    failed_count = 0

    try:
        for idx, (origin, dest) in enumerate(pending_pairs):
            pair_num = idx + len(processed_pairs) + 1
            print(f"[{pair_num}/{len(pairs)}] {origin} → {dest}...", end=" ", flush=True)

            try:
                start_time = time.time()
                data = fetch_fastest_connection(origin, dest)
                elapsed = time.time() - start_time

                if data:
                    print(f"{data['min_duration_minutes']} min (retrieved in {elapsed} seconds)")
                    batch_results.append(data)
                    api_calls_count += 1
                else:
                    print(f"no route found (retrieved in {elapsed} seconds)")

                    # if no route was found, still add it to output dataset.
                    # this is so that the scraping does not try again for this city combination when restarted
                    data = {
                        "origin_id": origin,
                        "destination_id": dest,
                        "origin_name": "",
                        "destination_name": "",
                        "min_duration_minutes": -1,
                        "transfers": 0,
                        "trains": "",
                        "check_date": get_next_representative_weekday().strftime("%Y-%m-%d")
                    }
                    batch_results.append(data)

                    api_calls_count += 1
                    failed_count += 1

                # batch save every N successful calls
                if api_calls_count >= batch_size:
                    batch_df = pd.DataFrame(batch_results)

                    # append to CSV or create if doesn't exist yet
                    if scraping_raw_output_file.exists():
                        batch_df.to_csv(scraping_raw_output_file, mode='a', header=False, index=False)
                    else:
                        batch_df.to_csv(scraping_raw_output_file, mode='w', header=True, index=False)

                    print(f"\nintermittent save: saved {api_calls_count} routes to {scraping_raw_output_file.name}")

                    batch_results = []
                    api_calls_count = 0

            except Exception as e:
                print(f"error: {e}")
                failed_count += 1
                time.sleep(2) # let's chill a bit after error to be safe
                continue

        # final save of remaining results
        if batch_results:
            batch_df = pd.DataFrame(batch_results)
            if scraping_raw_output_file.exists():
                batch_df.to_csv(scraping_raw_output_file, mode='a', header=False, index=False)
            else:
                batch_df.to_csv(scraping_raw_output_file, mode='w', header=True, index=False)
            print(f"\nsaved {len(batch_results)} remaining routes")

        print(f"scraping complete!")

    except KeyboardInterrupt:
        print(f"interrupted by user")
        # save any remaining batch data before exiting
        if batch_results:
            batch_df = pd.DataFrame(batch_results)
            if scraping_raw_output_file.exists():
                batch_df.to_csv(scraping_raw_output_file, mode='a', header=False, index=False)
            else:
                batch_df.to_csv(scraping_raw_output_file, mode='w', header=True, index=False)
            print(f"saved {len(batch_results)} routes before exit")
        raise

# Run scraping with intermittent saves

Note: as this notebook was refactored as not to include unnecessary code, the full output is not included here.
Overall scraping took approximately 3 hours.

In [ ]:
scrape_train_routes_with_checkpoints(cities_stations['eva_id'].tolist(), batch_size=10)

# Remove duplicates from saved data

In [ ]:
df = pd.read_csv(scraping_raw_output_file)

df = df.drop_duplicates()

df.to_csv(scraping_cleaned_output_file, index=False)

# Add back cities to routes data

Match up train stations with city names again, for easier data matching down the line

In [ ]:
df_routes = pd.read_csv(scraping_cleaned_output_file)

df_routes.rename(columns={"origin_name": "origin_station_name", "destination_name": "destination_station_name"}, inplace=True)

cities_stations_names_only = cities_stations[["city_name", "station_name"]]

cities_stations_origin = cities_stations.add_prefix("origin_")
cities_stations_destination = cities_stations.add_prefix("destination_")

df_routes_with_city_names = df_routes.merge(cities_stations_origin, on="origin_station_name")
df_routes_with_city_names = df_routes_with_city_names.merge(cities_stations_destination, on="destination_station_name")

df_routes_with_city_names

# Add a boolean column indicating if a route consists of >= 50% high-speed rail

## Load OpenStreetMap data


In [ ]:
osm_path = Path('Raw_Data/OSM_Data')
output_dir = Path('Raw_Data/Parquet_Data')
merged_path = Path('Raw_Data/EU_railways.parquet')

osm_path.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

###### (not necessary to run cell below since zip file already contains processed OpenStreetMap data)

In [ ]:
class RailwayExtractor(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.features = []

    def way(self, w):
        if w.tags.get("railway") != "rail":
            return

        coords = [(n.lon, n.lat) for n in w.nodes if n.location.valid()]
        if len(coords) < 2:
            return

        self.features.append({
            "geometry": LineString(coords),
            "maxspeed": w.tags.get("maxspeed"),
            "service": w.tags.get("service"),
            "usage": w.tags.get("usage"),
            "name": w.tags.get("name"),
        })

def extract_railways(pbf_path):
    handler = RailwayExtractor()
    handler.apply_file(pbf_path, locations=True)

    gdf = gpd.GeoDataFrame(
        handler.features,
        geometry="geometry",
        crs="EPSG:4326"
    )

    return gdf

def parse_maxspeed(val):
    if val is None:
        return None

    s = str(val).lower()

    m = re.search(r"\d+", str(val))
    if not m:
        return None

    speed = float(m.group())
    # Convert mph → km/h
    if "mph" in s:
        speed *= 1.609344
    return speed

def mark_highspeed(gdf):
    gdf = gdf.copy()

    gdf["maxspeed_num"] = gdf["maxspeed"].apply(parse_maxspeed)

    gdf["is_highspeed"] = (
        (gdf["maxspeed_num"] >= 200)
    )

    return gdf

for pbf in osm_path.glob("*.osm.pbf"):
    print(f"Processing {pbf.name}")

    gdf = extract_railways(pbf)
    gdf = mark_highspeed(gdf)

    out = output_dir / f"{pbf.stem}_rail.parquet"
    gdf.to_parquet(out)

## Merge all parquet files into one
###### (not necessary to run cell below since zip file already contains merged parquet file)

In [ ]:
files = output_dir.glob("*.parquet")

gdfs = [gpd.read_parquet(f) for f in files]

merged = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

merged.to_parquet(merged_path)

## Inspect merged railway data

In [ ]:
railways = gpd.read_parquet(merged_path)
print(railways.head) # note large number of rail segments
print(railways[railways['is_highspeed']].head)  # filter for is_highspeed

## Review distribution of data

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.hist(railways['maxspeed_num'], range=(0, 400))
ax1.set_title('Max Rail Speed Histogram')
ax1.set_xlabel('Max Speed (kmh)')
ax1.set_ylabel('Frequency')

ax2.boxplot(railways['maxspeed_num'].dropna())
ax2.set_title('Max Rail Speed Boxplot')
ax2.set_ylabel('Max Speed (kmh)')

fig.tight_layout()

## Inspect single high outlier

In [ ]:
railways.loc[railways['maxspeed_num'] > 300]

##### There is no rail line in Europe with a max speed of 980 kmh. The outlier is listed as a branch, meaning that it's supposed to be a railway with less traffic and lower speed than main lines. The coordinates show that this rail segment is directly next to a train station called Pečky, which is east of Prague. The most likely possibility is that this is a data tagging error and the actual max speed should be 98.0 kmh.

##### Correct the outlier speed and verify change. Dropping the outlier is not an option since it could break a route.

##### All other high-speed rail segments have correct max speeds.

In [ ]:
try:
    outlier_idx = railways.index[railways['maxspeed_num'] == 980][0]
    railways.at[outlier_idx, 'maxspeed'] = 98.0
    railways.at[outlier_idx, 'maxspeed_num'] = 98.0
    railways.at[outlier_idx, 'is_highspeed'] = False
    print(railways.iloc[outlier_idx]) # inspect corrected record
except IndexError:
    print("The outlier has already been removed")

## Inspect records with 0 kmh speeds


In [ ]:
railways.loc[railways['maxspeed_num'] == 0]

##### Considering 'usage' being None and the values for 'service', a speed of 0 makes sense for the above records.

## Build a rail network graph

In [ ]:
G = nx.Graph()

# convert to CRS 3035 since better for length calculations
rail_edges = railways[["geometry", "is_highspeed"]].copy()
rail_edges = rail_edges.to_crs(3035)

edge_sindex = rail_edges.sindex

for idx, row in rail_edges.iterrows():
    geom = row['geometry']
    if geom.geom_type == "MultiLineString":
        lines = geom.geoms
    else:
        lines = [geom]

    for line in lines:
        coords = list(line.coords)
        for u, v in zip(coords[:-1], coords[1:]):
            length = Point(u).distance(Point(v))
            G.add_edge(
                u, v,
                length=length,
                is_highspeed=row.is_highspeed
            )

## Snap stations to the graph

In [ ]:
def nearest_edge(point):
    res = edge_sindex.nearest(point) # 1st num always 0 (index of point), 2nd num index of rail edge
    idx = int(res[1, 0])
    return rail_edges.iloc[idx]

def project_to_edge(point, edge_geom):
    """Project a point onto a rail edge."""
    return edge_geom.interpolate(edge_geom.project(point))

def insert_station(G, station_point, edge_row):
    edge_geom = edge_row.geometry
    hsr = edge_row.is_highspeed

    projected = project_to_edge(station_point, edge_geom)

    coords = list(edge_geom.coords)

    # find segment where projection lies
    for u, v in zip(coords[:-1], coords[1:]):
        seg = LineString([u, v])
        if seg.distance(projected) < 0.01: # tolerance
            # remove original edge
            if G.has_edge(u, v):
                G.remove_edge(u, v)

            # add two new edges
            G.add_edge(
                u, projected.coords[0],
                length=Point(u).distance(projected),
                is_highspeed=hsr
            )
            G.add_edge(
                projected.coords[0], v,
                length=Point(v).distance(projected),
                is_highspeed=hsr
            )
            break

    # station node connects to projected point
    G.add_node(projected.coords[0])
    return projected.coords[0]

gpd_stations = gpd.GeoDataFrame(
    cities_with_stations,
    geometry=gpd.points_from_xy(
        cities_with_stations.longitude_y, cities_with_stations.latitude_y
    ),
    crs="EPSG:4326"
)

# project to CRS 3035
gpd_stations = gpd_stations.to_crs(3035)

# build lookup dictionary
station_to_node = {}

for _, row in gpd_stations.iterrows():
    pt = row.geometry
    edge = nearest_edge(pt)
    node = insert_station(G, pt, edge)
    station_to_node[row.station_name] = node

## Compute path & high-speed rail percentage per route
##### Approximate runtime: 3 hrs

In [123]:
def hsr_share_and_length(origin_node, dest_node):
    """length is given in meters"""
    path = nx.shortest_path(G, origin_node, dest_node, weight="length")

    total = 0
    hsr = 0

    for u, v in zip(path[:-1], path[1:]):
        edge = G[u][v]
        total += edge["length"]
        if edge["is_highspeed"]:
            hsr += edge["length"]

    hsr_share = hsr / total if total > 0 else 0
    return hsr_share, total

df_routes_with_city_names[["hsr_share", "route_length"]] = df_routes_with_city_names.apply(
    lambda r: pd.Series(
        hsr_share_and_length(
            station_to_node[r.origin_station_name],
            station_to_node[r.destination_station_name]
        )
    ),
    axis=1
)

## Determine threshold value for categorizing a route as using high-speed rail

The target value is 50%. However, the method of using shortest distance between origin and destination to determine the route is flawed. The shortest distance between the two points may not necessarily be the route taken by the train. Fine-grained information on a train's route is needed for the calculation, but this information is not available.

Manual inspection of a few routes using [DB](https://int.bahn.de/en/) and [OpenRailywayMap](https://www.openrailwaymap.org/) has shown that the current method underestimates the percentage of high-speed rail utilized. This is because the high-speed rail routes may not be the shortest distance between two points. This also indicates that the method underestimates the route distance.

Lowering the threshold was used to account for this discrepancy. The value used was determined as follows:
1. Filter df_routes_with_city_names for records with 'hsr_share' values between [0.45, 0.50).
2. Randomly sample five of those records.
3. Manually inspect each route using DB and OpenRailwayMap and visually judge if more than 50% of the route could consist of high-speed rail (time-consuming). If there appears to be a section of high-speed rail line that the train could reasonably take (may not be the shortest), then the train is assumed to take that section.
4. If at least one route does not seem to consist of 50% high-speed rail, set the threshold to the upper limit of the window.
5. Else go to step 1 and lower the window by 0.05.

The first route to not meet the visual threshold was from Zagreb Glavni kolodvor to Roma Tiburtina, with an hsr share of 36%. Therefore, set the threshold to 40%.

Unfortunately, some high-speed routes still fail to meet the threshold. An example is Vienna to Leipzig, which was calculated to have only a 7% share of HSR. Visual inspection shows that the actual route taken by the train is above 50% HSR. However, the shortest route touches almost no HSR.

In [124]:
routes_slice = df_routes_with_city_names.query("hsr_share >= 0.35 and hsr_share < 0.40")
routes_slice = routes_slice.dropna(subset=['origin_id', 'destination_id'])
sample = routes_slice.sample(5, random_state=42)
sample.head()

,origin_id,destination_id,origin_station_name,destination_station_name,min_duration_minutes,transfers,trains,check_date,origin_city_name,origin_eva_id,origin_latitude,origin_longitude,destination_city_name,destination_eva_id,destination_latitude,destination_longitude,hsr_share,route_length
1708,8400058,8000105,Amsterdam Centraal,Frankfurt(Main)Hbf,247,0,NaN,2026-01-06,Amsterdam,8400058,52.379190,4.899431,Frankfurt,8000105,50.106682,8.662828,0.359660,4.309365e+05
1955,7400002,8000098,Stockholm Central,Essen Hbf,967,4,NaN,2026-01-06,Stockholm,7400002,59.330014,18.057630,Essen,8000098,51.451355,7.014793,0.364760,1.491167e+06
3,8103000,8000085,Wien Hbf,Düsseldorf Hbf,499,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Düsseldorf,8000085,51.219962,6.794319,0.373210,9.458476e+05
1413,7800020,8300262,Zagreb Glavni kolodvor,Roma Tiburtina,795,4,NaN,2026-01-06,Zagreb,7800020,45.804438,15.978705,Rome,8300262,41.911070,12.531460,0.364917,9.025640e+05
342,8011160,8000152,Berlin Hbf,Hannover Hbf,93,0,NaN,2026-01-06,Berlin,8011160,52.525592,13.369545,Hannover,8000152,52.376761,9.741021,0.354796,2.553100e+05


In [125]:
df_routes_with_city_names["uses_hsr"] = df_routes_with_city_names["hsr_share"] >= 0.4
df_routes_with_city_names = df_routes_with_city_names.drop(columns='hsr_share', axis=1) # drop column to avoid confusion down the line

num_hsr_routes = len(df_routes_with_city_names[df_routes_with_city_names['uses_hsr'] == True])
print(f'Of the {len(df_routes_with_city_names)} routes, there are {num_hsr_routes} hsr routes.')

Of the 1646 routes, there are 233 hsr routes.


In [ ]:
df_routes_with_city_names.to_csv(scraping_cleaned_output_file, index=False)